## Задачи:
1) собрать геном
2) найти штаммы E. coli с максимальным сходством
3) аннотировать геном
4) найти отличающиеся гены у E. coli X
5) как E. coli X эволюционировала?
6) как E. coli X стала патогенной?

In [ ]:
#1. fastqc - анализ качества сборки
fastqc -o . /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/raw_data/SRR292678sub_S1_L001_R1_001.fastq.gz /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/raw_data/SRR292678sub_S1_L001_R2_001.fastq.gz
#в папке fastqc_results

In [ ]:
#3. sapdes
#SPAdes genome assembler v3.15.5
spades.py --test

spades.py \
  -1 /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/raw_data/SRR292678sub_S1_L001_R1_001.fastq.gz \
  -2 /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/raw_data/SRR292678sub_S1_L001_R2_001.fastq.gz \
  -o .spades
  -t 6 -m 16 \
  --careful

#оценка качества сборки quast
#QUAST v5.3.0
quast.py \
  /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/results/.spades/contigs.fasta \
  /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/results/.spades/scaffolds.fasta \
  -o /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/results/quast_report


In [ ]:
#4. Сборка с pacbio
#установка pacbio файла
fasterq-dump --split-3 SRR1980037

#запуск spades
spades.py \
  -1 /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/raw_data/SRR292678sub_S1_L001_R1_001.fastq.gz \
  -2 /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/raw_data/SRR292678sub_S1_L001_R2_001.fastq.gz \
  --pacbio /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/raw_data/SRR1980037.fastq \
  -o /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/results/spades_hybrid \
  -t 6 -m 16 \
  --careful

#оценка качества сборки quast
quast.py \
  /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/results/spades_hybrid/contigs.fasta \
  /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/results/spades_hybrid/scaffolds.fasta \
  -o /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/results/quast_report_hybrid

In [ ]:
#5. аннотация Prokka
#prokka 1.15.6
prokka /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/results/spades_hybrid/scaffolds.fasta \
  --outdir /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/results/prokka_results \
  --prefix annotation_SRR292678 \
  --cpus 8

In [ ]:
#6. Поиск ближайшего организма barrnap и Blast
#поиск 16s рРНК
barrnap /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/results/spades_hybrid/scaffolds.fasta > rrna.gff
grep "16S" rrna.gff > 16s.gff
#вытаскивание последовательности
bedtools getfasta \
  -fi /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab3/results/spades_hybrid/scaffolds.fasta \
  -bed 16s.gff \
  -fo 16s.fasta

blast  
веб версия, нуклеотидный бласт NCBI http://blast.ncbi.nlm.nih.gov
файл 16s.fasta (результат bedtools)  
настройки: организм - Escherichia coli в поле, база данных - refseq_genomes, PDAT - 1900/01/01:2011/01/01[PDAT]
fasta файл последовательности схожей E. coli скачала в папку raw_data

поиск генов вирулентности (синтез токсина) с помощью выравнивания с референсом Blast  - Mauve 
поиск генов вирулентности - веб версия ResFinder https://genepi.food.dtu.dk/resfinder